In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import ExtraTreesClassifier

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import lightgbm as lgb

In [32]:
train=pd.read_csv(r"F:\playground-series-s6e6\train.csv")
test=pd.read_csv(r"F:\playground-series-s6e6\test.csv")

In [33]:
train.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


In [34]:
test.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
0,577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
1,577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
2,577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
3,577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
4,577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence


In [35]:
print(train.shape)
print(test.shape)

(577347, 12)
(247435, 11)


In [36]:
TARGET="class"
ID_COL="id"

train_ids=train[ID_COL]
test_ids=test[ID_COL]

X=train.drop(columns=[TARGET, ID_COL])
y=train[TARGET]
X_test=test.drop(columns=[ID_COL])

In [37]:
target_encoder=LabelEncoder()
y_encoded=target_encoder.fit_transform(y)

In [ ]:
def feature_set_5(df):

    df = df.copy()
    # feature set 1
    df["u_g"]=df["u"] - df["g"]
    df["g_r"]=df["g"] - df["r"]
    df["r_i"]=df["r"] - df["i"]
    df["i_z"]=df["i"] - df["z"]
    df["u_r"]=df["u"] - df["r"]
    df["u_i"]=df["u"] - df["i"]
    df["u_z"]=df["u"] - df["z"]
    df["g_i"]=df["g"] - df["i"]
    df["g_z"]=df["g"] - df["z"]
    df["r_z"]=df["r"] - df["z"]

    # feature set 2
    eps=1e-6
    df["log_redshift"]=np.log1p(df["redshift"])
    df["sqrt_redshift"]=np.sqrt(np.clip(df["redshift"], 0, None))
    df["redshift_sq"]=(df["redshift"]**2)
    df["redshift_u"]=(df["redshift"] * df["u"])
    df["redshift_g"]=(df["redshift"] * df["g"])
    df["redshift_r"]=(df["redshift"] * df["r"])
    df["redshift_i"]=(df["redshift"] * df["i"])
    df["redshift_z"]=(df["redshift"] * df["z"])
    df["redshift_u_g"]=(df["redshift"] * df["u_g"])
    df["redshift_g_r"]=(df["redshift"] * df["g_r"])
    df["redshift_r_i"]=(df["redshift"] * df["r_i"])
    df["redshift_i_z"]=(df["redshift"] * df["i_z"])
    df["redshift_div_u"]=(df["redshift"] / (np.abs(df["u"]) + eps))
    df["redshift_div_g"]=(df["redshift"] / (np.abs(df["g"]) + eps))
    df["redshift_div_r"]=(df["redshift"] / (np.abs(df["r"]) + eps))
    df["redshift_div_i"]=(df["redshift"] / (np.abs(df["i"]) + eps))
    df["redshift_div_z"]=(df["redshift"] / (np.abs(df["z"]) + eps))
    
    # FEATURE SET 3
    df["curv_1"]=(df["u_g"] - df["g_r"])
    df["curv_2"]=(df["g_r"] - df["r_i"])
    df["curv_3"]=(df["r_i"] - df["i_z"])
    mag_cols=["u","g","r","i","z"]
    df["mag_std"]=(df[mag_cols].std(axis=1))
    df["mag_range"]=(df[mag_cols].max(axis=1))-(df[mag_cols].min(axis=1))
    df["alpha_sin"]=np.sin(np.radians(df["alpha"]))
    df["alpha_cos"]=np.cos(np.radians(df["alpha"]))
    df["delta_sin"]=np.sin(np.radians(df["delta"]))
    df["delta_cos"]=np.cos(np.radians(df["delta"]))
    
    # FEATURE SET 4
    alpha_rad = np.radians(df["alpha"])
    delta_rad = np.radians(df["delta"])

    # df["coord_x"]=(np.cos(delta_rad) * np.cos(alpha_rad))
    # df["coord_y"]=(np.cos(delta_rad) * np.sin(alpha_rad))
    # df["coord_z"]=(np.sin(delta_rad))
    df["redshift_zone"] = pd.cut(df["redshift"],bins=[-np.inf,0.0,0.15,0.50,1.0,1.3,np.inf],labels=False)
    z=np.asarray(df["redshift"],dtype=float)
    d_L=(299792.458 / 70.0) * z * (1 + 0.775*z)
    mu=np.where(z > 1e-4,5*np.log10(d_L*1e6) - 5,0.0)
    df["abs_mag_r"]=(df["r"] - mu)
    r_dist = np.log1p(np.clip(df["redshift"], 0, None))
    df["phys_x"] = (r_dist * np.cos(delta_rad) * np.cos(alpha_rad))
    df["phys_y"] = (r_dist *np.cos(delta_rad) *np.sin(alpha_rad))
    df["phys_z"] = (r_dist *np.sin(delta_rad))

    # FEATURE SET 5
    # Mean brightness
    df["mag_mean"] = df[mag_cols].mean(axis=1)

    # Approximate astronomical color-space loci
    STAR_GR=0.55
    STAR_RI=0.20

    QSO_GR=0.20
    QSO_RI=0.10

    GAL_GR=1.10
    GAL_RI=0.45

    # Distance to STAR locus
    df["stellar_locus_dist"]=np.sqrt((df["g_r"] - STAR_GR)**2 + (df["r_i"] - STAR_RI)**2)

    # Distance to QSO locus
    df["qso_locus_dist"]=np.sqrt((df["g_r"] - QSO_GR)**2 + (df["r_i"] - QSO_RI)**2)

    # Distance to GALAXY locus
    df["galaxy_locus_dist"]=np.sqrt((df["g_r"] - GAL_GR)**2 + (df["r_i"] - GAL_RI)**2)

    # Relative separation features
    df["star_vs_qso"]=(df["qso_locus_dist"] - df["stellar_locus_dist"])
    df["star_vs_gal"]=(df["galaxy_locus_dist"] - df["stellar_locus_dist"])
    df["qso_vs_gal"]=(df["galaxy_locus_dist"] - df["qso_locus_dist"])

    return df

In [39]:
X_fs5=feature_set_5(X)
X_test_fs5=feature_set_5(X_test)

print(X_fs5.shape)
print(X_test_fs5.shape)
print(X_fs5.columns.to_list())
print(X_test_fs5.columns.to_list())

C:\Users\WINDOWS 11\AppData\Local\Temp\ipykernel_1852\1705619861.py:58: RuntimeWarning: invalid value encountered in log10
  mu=np.where(z > 1e-4,5*np.log10(d_L*1e6) - 5,0.0)


(577347, 58)
(247435, 58)
['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population', 'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'u_i', 'u_z', 'g_i', 'g_z', 'r_z', 'log_redshift', 'sqrt_redshift', 'redshift_sq', 'redshift_u', 'redshift_g', 'redshift_r', 'redshift_i', 'redshift_z', 'redshift_u_g', 'redshift_g_r', 'redshift_r_i', 'redshift_i_z', 'redshift_div_u', 'redshift_div_g', 'redshift_div_r', 'redshift_div_i', 'redshift_div_z', 'curv_1', 'curv_2', 'curv_3', 'mag_std', 'mag_range', 'alpha_sin', 'alpha_cos', 'delta_sin', 'delta_cos', 'redshift_zone', 'abs_mag_r', 'phys_x', 'phys_y', 'phys_z', 'mag_mean', 'stellar_locus_dist', 'qso_locus_dist', 'galaxy_locus_dist', 'star_vs_qso', 'star_vs_gal', 'qso_vs_gal']
['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population', 'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'u_i', 'u_z', 'g_i', 'g_z', 'r_z', 'log_redshift', 'sqrt_redshift', 'redshift_sq', 'redshift_u', 'redshift_g', 'redshift_r', '

C:\Users\WINDOWS 11\AppData\Local\Temp\ipykernel_1852\1705619861.py:58: RuntimeWarning: invalid value encountered in log10
  mu=np.where(z > 1e-4,5*np.log10(d_L*1e6) - 5,0.0)


In [ ]:
def target_encode_multi(train_df,test_df,col,target_col,smoothing=20):

    classes=sorted(train_df[target_col].unique())
    global_probs=(train_df[target_col].value_counts(normalize=True))

    agg=(train_df.groupby(col)[target_col].value_counts(normalize=True).unstack(fill_value=0))
    counts=(train_df.groupby(col)[target_col].count())

    for cls in classes:
        if cls not in agg.columns:
            agg[cls]=0.0

        agg[cls]=(agg[cls] * counts + smoothing * global_probs.get(cls, 0)) / (counts + smoothing)
        train_df[f"{col}_{cls}_prob"]=(train_df[col].map(agg[cls]))
        test_df[f"{col}_{cls}_prob"]=(test_df[col].map(agg[cls]))

    return train_df,test_df

In [ ]:
train_te=X_fs5.copy()
train_te["class"]=y
test_te=X_test_fs5.copy()

In [ ]:
train_te,test_te=target_encode_multi(
    train_te,
    test_te,
    "spectral_type",
    "class",
    smoothing=20)

train_te,test_te=target_encode_multi(
    train_te,
    test_te,
    "galaxy_population",
    "class",
    smoothing=20)

In [ ]:
train_te=train_te.drop(columns=["spectral_type","galaxy_population"])
test_te=test_te.drop(columns=["spectral_type","galaxy_population"])

In [ ]:
drop_cols=["spectral_type_STAR_prob","galaxy_population_STAR_prob"]
train_te=train_te.drop(columns=drop_cols)
test_te=test_te.drop(columns=drop_cols)

In [ ]:
X_model=train_te.drop(columns=["class"])
X_test_model=test_te

print(X_model.shape)
print(X_test_model.shape)
print(X_model.columns.tolist())

(577347, 60)
(247435, 60)
['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'u_i', 'u_z', 'g_i', 'g_z', 'r_z', 'log_redshift', 'sqrt_redshift', 'redshift_sq', 'redshift_u', 'redshift_g', 'redshift_r', 'redshift_i', 'redshift_z', 'redshift_u_g', 'redshift_g_r', 'redshift_r_i', 'redshift_i_z', 'redshift_div_u', 'redshift_div_g', 'redshift_div_r', 'redshift_div_i', 'redshift_div_z', 'curv_1', 'curv_2', 'curv_3', 'mag_std', 'mag_range', 'alpha_sin', 'alpha_cos', 'delta_sin', 'delta_cos', 'redshift_zone', 'abs_mag_r', 'phys_x', 'phys_y', 'phys_z', 'mag_mean', 'stellar_locus_dist', 'qso_locus_dist', 'galaxy_locus_dist', 'star_vs_qso', 'star_vs_gal', 'qso_vs_gal', 'spectral_type_GALAXY_prob', 'spectral_type_QSO_prob', 'galaxy_population_GALAXY_prob', 'galaxy_population_QSO_prob']


In [ ]:
mi_scores=mutual_info_classif(X_model,y_encoded,random_state=42)

mi_df=pd.DataFrame({"Feature": X_model.columns,"MI":mi_scores})
mi_df.sort_values("MI",ascending=False).head(60)

,Feature,MI
30,redshift_div_u,0.525463
31,redshift_div_g,0.523478
32,redshift_div_r,0.516959
7,redshift,0.514987
18,log_redshift,0.514986
19,sqrt_redshift,0.514804
20,redshift_sq,0.514693
33,redshift_div_i,0.512378
25,redshift_z,0.511831
34,redshift_div_z,0.509394


In [ ]:
et=ExtraTreesClassifier(n_estimators=500,random_state=42,n_jobs=-1)
et.fit(X_model,y)

et_df = pd.DataFrame({
    "Feature": X_model.columns,
    "Importance": et.feature_importances_
})

et_df.sort_values("Importance",ascending=False).head(60)

,Feature,Importance
49,spectral_type_GALAXY_prob,0.068860
44,redshift_zone,0.059482
50,spectral_type_QSO_prob,0.054490
19,sqrt_redshift,0.041138
52,galaxy_population_QSO_prob,0.037060
18,log_redshift,0.036600
39,mag_range,0.035974
51,galaxy_population_GALAXY_prob,0.035955
7,redshift,0.030382
38,mag_std,0.028900


In [ ]:
lgbm_imp_model=LGBMClassifier(random_state=42,verbose=-1)

lgbm_imp_model.fit(X_model,y_encoded)
imp_df = pd.DataFrame({"Feature": X_model.columns,"Importance": lgbm_imp_model.feature_importances_})
imp_df.sort_values("Importance",ascending=False).head(60)

,Feature,Importance
43,delta_cos,774
1,delta,745
40,alpha_sin,648
41,alpha_cos,645
0,alpha,387
6,z,375
16,g_z,339
45,abs_mag_r,330
2,u,305
3,g,288


In [ ]:
FINAL_PARAMS = {
    "n_estimators": 3000,
    "learning_rate": 0.01,
    "max_depth": 12,
    "num_leaves": 127,
    "min_child_samples": 5,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "min_gain_to_split": 0.01,
    "class_weight": "balanced",
    "random_state": 42,
    "verbose": -1,
}

model_lgbm_4=LGBMClassifier(**FINAL_PARAMS)

model_lgbm_4.fit(X_model,y_encoded,callbacks=[lgb.log_evaluation(period=500)])
test_predictions = model_lgbm_4.predict(X_test_model)
test_predictions = target_encoder.inverse_transform(test_predictions)

submission_df = pd.DataFrame({"id": test_ids,"class": test_predictions})
submission_df.to_csv("submission_fset4_lgbm_out_trial.csv",index=False)
print("Submission file successfully created")
print(submission_df.head())
print("Trees built:",model_lgbm_4.booster_.num_trees())

Submission file successfully created
       id   class
0  577347  GALAXY
1  577348  GALAXY
2  577349  GALAXY
3  577350    STAR
4  577351  GALAXY
Trees built: 4695


In [ ]:
XGB_PARAMS = {
    'objective'        : 'multi:softprob',
    'num_class'        : 3,
    'n_estimators'     : 3000,
    'learning_rate'    : 0.01,
    'max_depth'        : 12,
    'subsample'        : 0.85,
    'colsample_bytree' : 0.85,
    #'reg_alpha'        : 0.05,
    #'reg_lambda'       : 1.0,
    'tree_method'      : 'hist',
    'random_state'     : 42,
    'n_jobs'           : -1,
    'device'           : 'cuda'
}

model_xgb_4 = XGBClassifier(**XGB_PARAMS)

model_xgb_4.fit(
    X_model,
    y_encoded
)

test_predictions = model_xgb_4.predict(X_test_model)

test_predictions = target_encoder.inverse_transform(
    test_predictions
)

submission_df = pd.DataFrame({
    "id": test_ids,
    "class": test_predictions
})

submission_df.to_csv(
    "submission_fset4_xgb_out.csv",
    index=False
)

print("Submission file successfully created")
print(submission_df.head())

Submission file successfully created
       id   class
0  577347  GALAXY
1  577348  GALAXY
2  577349  GALAXY
3  577350    STAR
4  577351  GALAXY
